In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError("event_blob_path is empty")

print("Event blob path:")
print(event_blob_path)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-6655203861910707>, line 6
      3 event_blob_path = dbutils.widgets.get("event_blob_path")
      5 if not event_blob_path:
----> 6     raise ValueError("event_blob_path is empty")
      8 print("Event blob path:")
      9 print(event_blob_path)

ValueError: event_blob_path is empty

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

Bronze path:
abfss://bronze@stcommercesrc.dfs.core.windows.net/


In [0]:
print("STEP 1 - Reading Bronze customer file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

display(df_bronze.limit(10))

STEP 1 - Reading Bronze customer file


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6655203861910711>, line 5
      1 print("STEP 1 - Reading Bronze customer file")
      3 df_bronze = spark.read.parquet(bronze_path)
----> 5 rows_read = df_bronze.count()
      7 print("Rows read from Bronze:", rows_read)
      9 display(df_bronze.limit(10))

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1922     query = self._plan.to_proto(self._session.client)
-> 1923     table, schema, self._execu

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    initcap,
    when,
    row_number
)

print("STEP 2 - Cleaning customer data")

df_clean = (
    df_bronze
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", trim(col("phone")))
    .withColumn("gender", upper(trim(col("gender"))))
    .withColumn("customer_status", upper(trim(col("customer_status"))))
)

print("Customer cleaning completed")
display(df_clean.limit(10))

STEP 2 - Cleaning customer data
Customer cleaning completed


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6655203861910763>, line 23
     12 df_clean = (
     13     df_bronze
     14     .withColumn("first_name", initcap(trim(col("first_name"))))
   (...)
     19     .withColumn("customer_status", upper(trim(col("customer_status"))))
     20 )
     22 print("Customer cleaning completed")
---> 23 display(df_clean.limit(10))

File /databricks/python_shell/lib/dbruntime/display.py:135, in Display.display(self, input, *args, **kwargs)
    133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
--> 135     if input.isStreaming:
    136         handleStreamingConnectDataFramePy4j(input, self.entry_point, kwargs)
    137     else:

File /usr/lib/python3.12/functools.py:995, in cached_property.__get__(self, instance, owner)
    993 val = cache.get(self.attrname, _NOT_FOUND)
    994 if val

In [0]:
from pyspark.sql.window import Window

print("STEP 3 - Deduplicating customers")

window_customer = Window.partitionBy(
    "customer_id"
).orderBy(
    col("updated_at").desc()
)

df_deduplicated = (
    df_clean
    .withColumn(
        "rn",
        row_number().over(window_customer)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

rows_after_dedup = df_deduplicated.count()

print("Rows after deduplication:", rows_after_dedup)

display(
    df_deduplicated.filter(col("customer_id") == 100)
)

STEP 3 - Deduplicating customers


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6655203861910764>, line 21
      5 window_customer = Window.partitionBy(
      6     "customer_id"
      7 ).orderBy(
      8     col("updated_at").desc()
      9 )
     11 df_deduplicated = (
     12     df_clean
     13     .withColumn(
   (...)
     18     .drop("rn")
     19 )
---> 21 rows_after_dedup = df_deduplicated.count()
     23 print("Rows after deduplication:", rows_after_dedup)
     25 display(
     26     df_deduplicated.filter(col("customer_id") == 100)
     27 )

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/conn

In [0]:
print("===== CUSTOMER TRANSFORMATION CHECK =====")

print("Bronze rows:", df_bronze.count())
print("Clean rows:", df_clean.count())
print("Deduplicated rows:", df_deduplicated.count())

print("Distinct customer IDs in Bronze:",
      df_bronze.select("customer_id").distinct().count())

print("Distinct customer IDs in Clean:",
      df_clean.select("customer_id").distinct().count())

print("Distinct customer IDs after dedup:",
      df_deduplicated.select("customer_id").distinct().count())

print("=========================================")

===== CUSTOMER TRANSFORMATION CHECK =====


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6655203861910834>, line 3
      1 print("===== CUSTOMER TRANSFORMATION CHECK =====")
----> 3 print("Bronze rows:", df_bronze.count())
      4 print("Clean rows:", df_clean.count())
      5 print("Deduplicated rows:", df_deduplicated.count())

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1922     query = self._plan.to_proto(self._session.client)
-> 1923     table, schema, self._execution_info = self.

In [0]:
print("STEP 4 - Checking duplicate customer IDs")

duplicate_count = (
    df_deduplicated
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Customer IDs still duplicated:", duplicate_count)

if duplicate_count != 0:
    raise ValueError(
        f"Deduplication failed. {duplicate_count} customer IDs are still duplicated."
    )

print("Duplicate check PASSED")

STEP 4 - Checking duplicate customer IDs


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6655203861910765>, line 8
      1 print("STEP 4 - Checking duplicate customer IDs")
      3 duplicate_count = (
      4     df_deduplicated
      5     .groupBy("customer_id")
      6     .count()
      7     .filter(col("count") > 1)
----> 8     .count()
      9 )
     11 print("Customer IDs still duplicated:", duplicate_count)
     13 if duplicate_count != 0:

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructTyp

In [0]:
print("STEP 5 - Validating customer records")

df_validated = (
    df_deduplicated
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())
    .filter(trim(col("email")) != "")
    .filter(col("customer_status").isin("ACTIVE", "INACTIVE"))
)

rows_after_validation = df_validated.count()

print("Rows after validation:", rows_after_validation)

display(
    df_validated.limit(10)
)

STEP 5 - Validating customer records


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6655203861910810>, line 4
      1 print("STEP 5 - Validating customer records")
      3 df_validated = (
----> 4     df_deduplicated
      5     .filter(col("customer_id").isNotNull())
      6     .filter(col("email").isNotNull())
      7     .filter(trim(col("email")) != "")
      8     .filter(col("customer_status").isin("ACTIVE", "INACTIVE"))
      9 )
     11 rows_after_validation = df_validated.count()
     13 print("Rows after validation:", rows_after_validation)

NameError: name 'df_deduplicated' is not defined

In [0]:
rejected_rows = rows_after_dedup - rows_after_validation

print("Rows after deduplication:", rows_after_dedup)
print("Rows after validation:", rows_after_validation)
print("Rows rejected:", rejected_rows)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6655203861910811>, line 1
----> 1 rejected_rows = rows_after_dedup - rows_after_validation
      3 print("Rows after deduplication:", rows_after_dedup)
      4 print("Rows after validation:", rows_after_validation)

NameError: name 'rows_after_dedup' is not defined

In [0]:
print("STEP 6 - Checking customer ID NULLs")

null_customer_ids = (
    df_bronze
    .filter(col("customer_id").isNull())
    .count()
)

print("Bronze rows with NULL customer_id:", null_customer_ids)

In [0]:
print("STEP 7 - Checking duplicate customer IDs")

duplicate_ids = (
    df_clean
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

print("Number of customer IDs with duplicates:", duplicate_ids.count())

display(
    duplicate_ids.orderBy(col("count").desc())
)

In [0]:
print("STEP 6 - Categorizing rejected customer records")

df_rejected = (
    df_deduplicated
    .withColumn(
        "rejection_reason",
        when(
            col("customer_id").isNull(),
            "NULL customer_id"
        )
        .when(
            col("email").isNull(),
            "NULL email"
        )
        .when(
            trim(col("email")) == "",
            "EMPTY email"
        )
        .when(
            ~col("customer_status").isin("ACTIVE", "INACTIVE"),
            "INVALID customer_status"
        )
        .otherwise("UNKNOWN")
    )
    .filter(col("rejection_reason") != "UNKNOWN")
)

display(
    df_rejected
    .groupBy("rejection_reason")
    .count()
    .orderBy(col("count").desc())
)

STEP 6 - Categorizing rejected customer records


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5464767788261973>, line 4
      1 print("STEP 6 - Categorizing rejected customer records")
      3 df_rejected = (
----> 4     df_deduplicated
      5     .withColumn(
      6         "rejection_reason",
      7         when(
      8             col("customer_id").isNull(),
      9             "NULL customer_id"
     10         )
     11         .when(
     12             col("email").isNull(),
     13             "NULL email"
     14         )
     15         .when(
     16             trim(col("email")) == "",
     17             "EMPTY email"
     18         )
     19         .when(
     20             ~col("customer_status").isin("ACTIVE", "INACTIVE"),
     21             "INVALID customer_status"
     22         )
     23         .otherwise("UNKNOWN")
     24     )
     25     .filter(col("rejection_reason") != "UNKNO

In [0]:
print("Total rejected records by reason:")

rejection_summary = (
    df_rejected
    .groupBy("rejection_reason")
    .count()
)

display(rejection_summary)

print(
    "Total rejected:",
    rejection_summary.agg({"count": "sum"}).collect()[0][0]
)

In [0]:
print("STEP 7 - Checking invalid customer statuses")

invalid_statuses = (
    df_deduplicated
    .filter(
        col("customer_status").isNull()
        | (~col("customer_status").isin("ACTIVE", "INACTIVE"))
    )
    .groupBy("customer_status")
    .count()
    .orderBy(col("count").desc())
)

display(invalid_statuses)

In [0]:
print("STEP 8 - Inspecting NULL email record")

display(
    df_rejected
    .filter(col("rejection_reason") == "NULL email")
)

In [0]:
print("STEP 9 - Applying corrected customer validation")

df_validated = (
    df_deduplicated
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())
    .filter(trim(col("email")) != "")
    .filter(
        col("customer_status").isin(
            "ACTIVE",
            "INACTIVE",
            "SUSPENDED"
        )
    )
)

rows_after_validation = df_validated.count()

print("Rows after deduplication:", rows_after_dedup)
print("Rows after validation:", rows_after_validation)
print("Rows rejected:", rows_after_dedup - rows_after_validation)

display(df_validated.limit(10))

In [0]:
print("STEP 10 - Creating rejected customer records")

df_rejected_customers = (
    df_deduplicated
    .withColumn(
        "rejection_reason",
        when(
            col("customer_id").isNull(),
            "NULL customer_id"
        )
        .when(
            col("email").isNull(),
            "NULL email"
        )
        .when(
            trim(col("email")) == "",
            "EMPTY email"
        )
        .when(
            ~col("customer_status").isin(
                "ACTIVE",
                "INACTIVE",
                "SUSPENDED"
            ),
            "INVALID customer_status"
        )
        .otherwise("UNKNOWN")
    )
    .filter(col("rejection_reason") != "UNKNOWN")
)

rejected_count = df_rejected_customers.count()

print("Rejected customer records:", rejected_count)

display(df_rejected_customers)

In [0]:
print("STEP 11 - Writing validated customers to Silver")

silver_customers_path = (
    "abfss://silver@stcommercesrc.dfs.core.windows.net/"
    "customers_event/"
)

(
    df_validated
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_customers_path)
)

print("Silver customers written successfully")
print("Rows written:", df_validated.count())
print("Silver path:", silver_customers_path)

In [0]:
print("STEP 12 - Verifying Silver customers")

df_silver_check = spark.read.format("delta").load(
    "abfss://silver@stcommercesrc.dfs.core.windows.net/customers_event/"
)

print("Silver row count:", df_silver_check.count())
print("Silver columns:", df_silver_check.columns)

display(df_silver_check.limit(10))

In [0]:
from delta.tables import DeltaTable

print("STEP 13 - Merging customers into Silver")

silver_table = DeltaTable.forName(
    spark,
    "ecommerce.silver.customers_external"
)

(
    silver_table.alias("target")
    .merge(
        df_validated.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Customer MERGE completed successfully")

In [0]:
print("STEP 14 - Verifying final Silver customers")

df_final_customers = spark.table(
    "ecommerce.silver.customers_external"
)

print("Final Silver customer count:", df_final_customers.count())

display(
    df_final_customers
    .filter(col("customer_id").isin(100, 25001, 108))
)

In [0]:
print("STEP 15 - Rebuilding customer exceptions using corrected rules")

df_rejected_corrected = (
    df_deduplicated
    .filter(
        col("customer_id").isNull()
        | col("email").isNull()
        | (trim(col("email")) == "")
        | (
            ~col("customer_status").isin(
                "ACTIVE",
                "INACTIVE",
                "SUSPENDED"
            )
        )
    )
    .withColumn(
        "rejection_reason",
        when(
            col("customer_id").isNull(),
            "NULL customer_id"
        )
        .when(
            col("email").isNull(),
            "NULL email"
        )
        .when(
            trim(col("email")) == "",
            "EMPTY email"
        )
        .otherwise("INVALID customer_status")
    )
)

print(
    "Corrected exception rows:",
    df_rejected_corrected.count()
)

display(df_rejected_corrected)

In [0]:
print("STEP 15 - Saving corrected customer exception")

(
    df_rejected_corrected
    .write
    .format("delta")
    .mode("overwrite")
    .save(
        "abfss://silver@stcommercesrc.dfs.core.windows.net/"
        "exceptions/customers/"
    )
)

print("Corrected customer exceptions saved")

print("Rows saved:", df_rejected_corrected.count())

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.silver.customer_exceptions
USING DELTA
LOCATION 'abfss://silver@stcommercesrc.dfs.core.windows.net/exceptions/customers/';

In [0]:
%sql
SELECT
    customer_id,
    rejection_reason,
    COUNT(*) AS record_count
FROM ecommerce.silver.customer_exceptions
GROUP BY
    customer_id,
    rejection_reason
ORDER BY customer_id;

customer_id,rejection_reason,record_count
108,NULL email,1


In [0]:
%sql
SELECT
    pipeline_name,
    table_name,
    run_id,
    event_blob_path,
    status,
    trigger_type,
    start_time
FROM ecommerce.control.pipeline_audit
ORDER BY start_time DESC
LIMIT 1;

pipeline_name,table_name,run_id,event_blob_path,status,trigger_type,start_time
PL_Master_Ecommerce,customersevent_test_customer_001.parquet,39980abf-2822-404f-ae1f-2ef90260433e,bronze/customersevent_test_customer_001.parquet,Started,BlobEventsTrigger,2026-10-04T11:19:26.249462Z


In [0]:
%sql
SELECT current_user();

current_user()
saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com


In [0]:
%sql
SELECT COUNT(*) AS silver_customer_count
FROM ecommerce.silver.customers_external;

silver_customer_count
24997


In [0]:
display(dbutils.fs.ls("abfss://bronze@stcommercesrc.dfs.core.windows.net/customers/data_da08fe20-4f3b-4f7c-8fd5-1d963d98e626_370a9fd9-0bb8-40b3-afb1-9696129c4bdc.parquet"))

path,name,size,modificationTime
abfss://bronze@stcommercesrc.dfs.core.windows.net/customers/data_da08fe20-4f3b-4f7c-8fd5-1d963d98e626_370a9fd9-0bb8-40b3-afb1-9696129c4bdc.parquet,data_da08fe20-4f3b-4f7c-8fd5-1d963d98e626_370a9fd9-0bb8-40b3-afb1-9696129c4bdc.parquet,307578,1790952783000


In [0]:
df_bn = spark.read.format('parquet')\
                    .option('inferSchema', 'true')\
                    .load("abfss://bronze@stcommercesrc.dfs.core.windows.net/customers/data_da08fe20-4f3b-4f7c-8fd5-1d963d98e626_370a9fd9-0bb8-40b3-afb1-9696129c4bdc.parquet")
df_bn.display()

customer_id,first_name,last_name,email,phone,gender,date_of_birth,registration_date,customer_status,created_at,updated_at
9,Farhan,Chada,wbiswas@example.net,+912782489638,OTHER,2007-02-23T00:00:00Z,2026-03-17T19:36:18Z,ACTIVE,2026-03-17T19:36:18Z,2026-03-20T19:36:18Z
11,Ekavir,Bhargava,ekanshbalay@example.com,+918347382997,FEMALE,1972-04-14T00:00:00Z,2026-09-03T17:27:53Z,ACTIVE,2026-09-03T17:27:53Z,2027-02-22T17:27:53Z
24,Ranveer,Kumar,sarinaarav@example.net,3561595148,OTHER,1976-10-18T00:00:00Z,2025-09-02T23:08:20Z,ACTIVE,2025-09-02T23:08:20Z,2026-07-10T23:08:20Z
41,Qasim,Handa,vedamalhotra@example.net,8941343524,FEMALE,1956-09-17T00:00:00Z,2026-01-08T17:41:04Z,ACTIVE,2026-01-08T17:41:04Z,2026-09-04T17:41:04Z
47,Upadhriti,Modi,gmagar@example.net,07190659401,OTHER,1971-04-30T00:00:00Z,2025-09-25T23:19:49Z,INACTIVE,2025-09-25T23:19:49Z,2026-02-06T23:19:49Z
49,Henry,Dyal,nisha74@example.net,08071545168,MALE,1955-10-30T00:00:00Z,2025-08-05T06:42:44Z,INACTIVE,2025-08-05T06:42:44Z,2026-01-04T06:42:44Z
59,Jagrati,Merchant,anthony64@example.org,0974395339,OTHER,1972-10-21T00:00:00Z,2025-10-06T17:46:51Z,ACTIVE,2025-10-06T17:46:51Z,2026-07-14T17:46:51Z
62,Aishani,Yadav,shajyoti@example.net,4965137098,OTHER,1978-01-26T00:00:00Z,2025-03-10T11:01:33Z,ACTIVE,2025-03-10T11:01:33Z,2026-03-10T11:01:33Z
66,Sarthak,Kalita,falgunidivan@example.net,+914228421020,MALE,1980-02-17T00:00:00Z,2026-09-20T09:31:33Z,ACTIVE,2026-09-20T09:31:33Z,2027-07-18T09:31:33Z
69,Ekta,Raj,sudiksha89@example.com,+911836736576,MALE,1984-01-22T00:00:00Z,2026-03-16T19:51:07Z,ACTIVE,2026-03-16T19:51:07Z,2026-04-01T19:51:07Z


In [0]:
df_bn.count()

4473

In [0]:
dbutils.jobs.taskValues.set(
    key="rows_read",
    value=int(rows_read)
)

dbutils.jobs.taskValues.set(
    key="rows_written",
    value=int(rows_after_validation)
)

print("Audit values prepared")
print("Rows read:", rows_read)
print("Rows written:", rows_after_validation)